# Week 1 EDA: the sampled application table

Work through the sections in order. Cells marked **PROVIDED** run as they are. For each
**TODO(student)** section, answer the question with code in the empty cell, then write one
or two sentences of what you see. Keep reusable code in `src/`; this notebook is for looking.

Run it from the repository root with `jupyter notebook` (or `jupyter lab`) after `python run.py sample`.

## 1. Load the sample  (PROVIDED)

In [ ]:
# PROVIDED
import sys
from pathlib import Path

import matplotlib.pyplot as plt

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

from src.data import load_application  # noqa: E402
from src.utils import load_config  # noqa: E402

config = load_config(ROOT / "config" / "config.yaml")
app = load_application(config)
print(app.shape)
app.head()

## 2. Target rate  (PROVIDED)

In [ ]:
# PROVIDED
counts = app["TARGET"].value_counts().sort_index()
print(counts)
print(f"Default rate: {app['TARGET'].mean():.2%}")
counts.plot.bar(title="TARGET (1 = payment difficulties)")
plt.show()

## 3. Missingness  (TODO(student))

Which columns have the most missing values? How many columns are more than half empty?
Do the most-missing columns seem to belong together (look at their names)?

In [ ]:
# reference solution
missing = app.isna().mean().sort_values(ascending=False)
print(f"Columns more than half empty: {(missing > 0.5).sum()} of {app.shape[1]}")
missing.head(20)

## 4. The DAYS_EMPLOYED anomaly  (TODO(student))

Look at the distribution of `DAYS_EMPLOYED`. Which value looks impossible, and how many
applicants have it? Is the default rate of those applicants different from everyone else's?
What does that tell you about how to handle it?

In [ ]:
# reference solution
sentinel = 365243
is_anom = app["DAYS_EMPLOYED"] == sentinel
print(f"Applicants with DAYS_EMPLOYED == {sentinel}: {is_anom.sum()} ({is_anom.mean():.1%})")
by_group = app.groupby(is_anom)["TARGET"].agg(["mean", "size"])
print(by_group.rename(index={False: "normal", True: "anomaly"}))
app.loc[~is_anom, "DAYS_EMPLOYED"].plot.hist(bins=50, title="DAYS_EMPLOYED without the sentinel")
plt.show()

## 5. Default rate by a categorical column  (TODO(student))

Pick one categorical column (for example `NAME_CONTRACT_TYPE`, `CODE_GENDER`, or
`OCCUPATION_TYPE`). Show the default rate and the number of applicants in each category.
Which differences would you trust, given how few applicants some categories have?

In [ ]:
# reference solution
for col in ["NAME_CONTRACT_TYPE", "CODE_GENDER", "OCCUPATION_TYPE"]:
    table = app.groupby(col, dropna=False)["TARGET"].agg(rate="mean", n="size")
    print(table.sort_values("n", ascending=False), end="\n\n")

## 6. Correlations with TARGET  (TODO(student))

Which numeric columns are most correlated with `TARGET` (positively and negatively)?
Remember the sign conventions: `DAYS_*` columns are negative numbers of days before the
application. Is a correlation of 0.1 large or small here?

In [ ]:
# reference solution
numeric = app.select_dtypes("number").drop(columns=["SK_ID_CURR"])
# Constant columns (e.g. FLAG_MOBIL in this sample) have no correlation (NaN): drop them.
corr = numeric.corr()["TARGET"].drop("TARGET").dropna().sort_values()
print("Most negative:\n", corr.head(8), "\n\nMost positive:\n", corr.tail(8))

## 7. Findings  (instructor reference)

- About 8% of the sampled applicants default, so accuracy would be misleading.
- Many housing-related columns are mostly empty; median imputation is a reasonable baseline.
- DAYS_EMPLOYED == 365243 marks a placeholder: all 897 such applicants in the sample are
  pensioners, and they default less often (about 4.9% vs 8.8%), so the baseline keeps a
  flag and treats the value itself as missing.
- EXT_SOURCE_* columns have the strongest (negative) correlation with TARGET.
- Small categories have very few defaults, so their rates are noisy.